<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TPU_H2E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
!pip install -q --no-deps "vllm-tpu==0.21.0"

import importlib.metadata as m, subprocess, sys
reqs = [r for r in m.requires("vllm-tpu")
        if "extra ==" not in r and not r.lower().startswith("nixl")]
subprocess.run([sys.executable, "-m", "pip", "install", *reqs])

In [2]:
import vllm
print(vllm.__version__)

0.21.0


In [ ]:
!pip install "tpu-inference==0.21.0" 2>&1 | tail -n 20

In [1]:
!pip list 2>/dev/null | grep -Ei "^vllm|tpu.inference|^jax "

jax                          0.9.2
tpu_inference                0.21.0
vllm-tpu                     0.21.0


In [2]:
import importlib.metadata as m, subprocess, sys, re

reqs = [r for r in m.requires("vllm-tpu")
        if "extra ==" not in r and not r.lower().startswith("nixl")]

for attempt in range(6):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *reqs],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ all dependencies installed")
        break
    blocked = re.search(r"Cannot uninstall (\S+)", r.stdout + r.stderr)
    if not blocked:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        break
    pkg = blocked.group(1)
    print(f"system package {pkg} blocks the install → reinstalling it with --ignore-installed")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-installed", pkg])

system package PyJWT blocks the install → reinstalling it with --ignore-installed
✅ all dependencies installed


In [ ]:
# ================= SILENCE EVERYTHING (must come before any other import) =================
import os, io, warnings, logging, contextlib
os.environ["VLLM_LOGGING_LEVEL"] = "CRITICAL"          # vLLM + tpu_inference logs
os.environ["TQDM_DISABLE"] = "1"                       # progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"       # Hugging Face download bars
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.CRITICAL)
for name in ("", "absl", "jax", "torchax", "transformers", "huggingface_hub"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["MODEL_IMPL_TYPE"] = "vllm"

import re
import vllm.utils.system_utils as _su
import vllm.distributed.parallel_state as _ps

@contextlib.contextmanager
def _no_suppress():
    yield
_su.suppress_stdout = _no_suppress   # Jupyter stdout has no fileno()
_ps.suppress_stdout = _no_suppress

from vllm import LLM, SamplingParams
from vllm.platforms import current_platform
logging.getLogger().setLevel(logging.CRITICAL)   # some libraries reset the root logger on import

# ---------------- prompt ----------------
def build_prompt(en: str) -> str:
    return (
        "Translate English to Hindi.\n\n"
        "English: The sky is blue.\n"
        "Hindi: आकाश नीला है।\n\n"
        "English: Artificial intelligence is transforming the world.\n"
        "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।\n\n"
        "English: Modern AI is powerful.\n"
        "Hindi: आधुनिक एआई शक्तिशाली है।\n\n"
        "English: Robust software is reliable.\n"
        "Hindi: मज़बूत सॉफ़्टवेयर विश्वसनीय होता है।\n\n"
        "English: The weather today is very beautiful.\n"
        "Hindi: आज का मौसम बहुत खूबसूरत है।\n\n"
        "English: Deep learning requires large datasets to function well.\n"
        "Hindi: डीप लर्निंग को अच्छी तरह काम करने के लिए बड़े डेटासेट की आवश्यकता होती है।\n\n"
        f"English: {en}\n"
        "Hindi:"
    )

# ---------------- model (loaded on the TPU by vLLM) ----------------
with contextlib.redirect_stderr(io.StringIO()):     # hide vLLM's own loading progress bar
    llm = LLM(
        model="sarvamai/sarvam-1",
        max_model_len=2048,
        max_num_seqs=8,
        max_num_batched_tokens=2048,
    )
text_model = llm      # rename: from here on only text_model refers to the model
del llm
print(f"✓ Sarvam-1 on {current_platform.get_device_name()}\n")

STOPS = ["\n", "English:", "Note:"]
full_params = SamplingParams(temperature=0.0, max_tokens=64, stop=STOPS)
next_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)

LATIN_WORD = re.compile(r"[A-Za-z]{3,}")
DEVANAGARI = re.compile(r"[\u0900-\u097F]")

def is_copied(text: str) -> bool:
    """True if the output still contains English words or no Hindi at all."""
    return bool(LATIN_WORD.search(text)) or not DEVANAGARI.search(text)

def best_hindi_piece(logprobs_dict) -> str:
    """Most likely candidate token that is Devanagari and has no English letters."""
    for lp in sorted(logprobs_dict.values(), key=lambda x: x.rank):
        piece = lp.decoded_token or ""
        if DEVANAGARI.search(piece) and not re.search(r"[A-Za-z]", piece):
            return piece
    return ""

def force_hindi(prompt: str, rounds: int = 4) -> str:
    """Steer the answer into Hindi: pick a Devanagari token, continue, repeat if English returns."""
    answer = ""
    for _ in range(rounds):
        nxt = text_model.generate([prompt + " " + answer if answer else prompt], next_params, use_tqdm=False)
        piece = best_hindi_piece(nxt[0].outputs[0].logprobs[0])
        if not piece:
            break
        answer = (answer + piece).strip() if answer else piece.strip()
        cont = text_model.generate([prompt + " " + answer], full_params, use_tqdm=False)[0].outputs[0].text
        candidate = (answer + cont).strip()
        if not LATIN_WORD.search(candidate):
            return candidate
        answer = LATIN_WORD.split(candidate)[0].strip()   # keep Hindi part, steer again
    return answer

def translate(sentences):
    prompts = [build_prompt(s) for s in sentences]
    outs = text_model.generate(prompts, full_params, use_tqdm=False)
    results = [o.outputs[0].text.strip() for o in outs]
    for i, r in enumerate(results):
        if is_copied(r):
            results[i] = force_hindi(prompts[i])
    return results

# ---------------- test ----------------
tests = [
    "Resilient AI is efficient.",
    "The children are playing in the park.",
    "Please close the door when you leave.",
]
for en, hi in zip(tests, translate(tests)):
    print(f"EN: {en}\nHI: {hi}\n")

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


✓ Sarvam-1 on TPU V6E

EN: Resilient AI is efficient.
HI: प्रतिरोधी एआई कुशल है।

EN: The children are playing in the park.
HI: बच्चे पार्क में खेल रहे हैं।

EN: Please close the door when you leave.
HI: कृपया दरवाज़ा बंद कर दीजिए।



## MODELS - MULTIMODAL

In [4]:
!pip install -U bitsandbytes>=0.46.1 -q

In [1]:
# ================= SILENCE (before any other import) =================
import os, io, re, gc, time, warnings, logging, contextlib
os.environ["VLLM_LOGGING_LEVEL"] = "CRITICAL"
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["MODEL_IMPL_TYPE"] = "vllm"
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.CRITICAL)
for n in ("", "absl", "jax", "torchax", "transformers", "huggingface_hub"):
    logging.getLogger(n).setLevel(logging.CRITICAL)

import numpy as np, requests, librosa, torch, jax, torchax
import torch.nn.functional as F
from io import BytesIO
from PIL import Image
import vllm.utils.system_utils as _su
import vllm.distributed.parallel_state as _ps
from vllm import LLM, SamplingParams
from vllm.platforms import current_platform
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          AutoProcessor, AutoModelForImageTextToText)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
logging.getLogger().setLevel(logging.CRITICAL)

@contextlib.contextmanager
def _no_suppress():
    yield
_su.suppress_stdout = _no_suppress   # Colab stdout has no fileno()
_ps.suppress_stdout = _no_suppress

# ================= TPU helpers + torchax patches (same math, supported ops) =================
env = torchax.default_env()

def tpu_hbm_gb():
    s = jax.local_devices()[0].memory_stats() or {}
    return s.get("bytes_in_use", 0) / 1024**3

def to_cpu(t):
    try:
        return t.to("cpu")
    except Exception:
        return torch.from_numpy(np.asarray(jax.device_get(t._elem)))

def _tup(x):
    return [x] if isinstance(x, int) else list(x)

def _conv1d(input, weight, bias=None, stride=1, padding=0, dilation=1, groups=1):
    return torch.ops.aten.convolution(input, weight, bias, _tup(stride), _tup(padding),
                                      _tup(dilation), False, [0], groups)
F.conv1d = _conv1d

_orig_dropout = F.dropout
F.dropout = lambda input, p=0.5, training=True, inplace=False: (
    input if (not training or p == 0.0) else _orig_dropout(input, p, training, inplace))

torch.Tensor.expand_as = lambda self, other: self.expand(other.shape)

def _masked_scatter(self, mask, source):
    mask = mask.expand(self.shape).reshape(-1)
    idx = (torch.cumsum(mask.to(torch.int32), 0) - 1).clamp(min=0)
    return torch.where(mask, source.reshape(-1)[idx], self.reshape(-1)).reshape(self.shape)
torch.Tensor.masked_scatter = _masked_scatter
# note: use torch.no_grad() (not inference_mode) with torchax

# ================= 1. TEXT: Sarvam-1 via vLLM (always loaded) =================
with contextlib.redirect_stderr(io.StringIO()):
    llm = LLM(model="sarvamai/sarvam-1", max_model_len=2048, max_num_seqs=8,
              max_num_batched_tokens=2048,
              gpu_memory_utilization=0.18)     # ~5.6 GB: leaves room for Gemma's peak
text_model = llm
del llm
print(f"✓ TEXT  Sarvam-1 on {current_platform.get_device_name()} | TPU HBM {tpu_hbm_gb():.1f} GB")

def build_prompt(en):
    return ("Translate English to Hindi.\n\n"
            "English: The sky is blue.\nHindi: आकाश नीला है।\n\n"
            "English: Artificial intelligence is transforming the world.\n"
            "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।\n\n"
            "English: Modern AI is powerful.\nHindi: आधुनिक एआई शक्तिशाली है।\n\n"
            "English: Robust software is reliable.\nHindi: मज़बूत सॉफ़्टवेयर विश्वसनीय होता है।\n\n"
            "English: The weather today is very beautiful.\nHindi: आज का मौसम बहुत खूबसूरत है।\n\n"
            "English: Deep learning requires large datasets to function well.\n"
            "Hindi: डीप लर्निंग को अच्छी तरह काम करने के लिए बड़े डेटासेट की आवश्यकता होती है।\n\n"
            f"English: {en}\nHindi:")

full_params = SamplingParams(temperature=0.0, max_tokens=64, stop=["\n", "English:", "Note:"])
next_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
LATIN_WORD, DEVANAGARI = re.compile(r"[A-Za-z]{3,}"), re.compile(r"[\u0900-\u097F]")

def _best_hindi_piece(lps):
    for lp in sorted(lps.values(), key=lambda x: x.rank):
        p = lp.decoded_token or ""
        if DEVANAGARI.search(p) and not re.search(r"[A-Za-z]", p):
            return p
    return ""

def _force_hindi(prompt, rounds=4):
    answer = ""
    for _ in range(rounds):
        nxt = text_model.generate([prompt + " " + answer if answer else prompt], next_params, use_tqdm=False)
        piece = _best_hindi_piece(nxt[0].outputs[0].logprobs[0])
        if not piece:
            break
        answer = (answer + piece).strip() if answer else piece.strip()
        cont = text_model.generate([prompt + " " + answer], full_params, use_tqdm=False)[0].outputs[0].text
        cand = (answer + cont).strip()
        if not LATIN_WORD.search(cand):
            return cand
        answer = LATIN_WORD.split(cand)[0].strip()
    return answer

def translate(sentences):
    prompts = [build_prompt(s) for s in sentences]
    res = [o.outputs[0].text.strip() for o in text_model.generate(prompts, full_params, use_tqdm=False)]
    return [_force_hindi(p) if (LATIN_WORD.search(r) or not DEVANAGARI.search(r)) else r
            for p, r in zip(prompts, res)]

# ================= 2. AUDIO: Whisper via torchax (on demand) =================
SR, CHUNK, AUDIO_MODEL = 16000, 30 * 16000, "openai/whisper-large-v3-turbo"
audio_model = audio_processor = None

def load_audio():
    global audio_model, audio_processor
    audio_processor = WhisperProcessor.from_pretrained(AUDIO_MODEL)
    audio_model = WhisperForConditionalGeneration.from_pretrained(
        AUDIO_MODEL, dtype=torch.bfloat16, attn_implementation="eager").eval()   # eager: avoid vLLM's attention kernel
    with env:
        audio_model.to("jax")
    print(f"✓ AUDIO Whisper loaded | TPU HBM {tpu_hbm_gb():.1f} GB")

def unload_audio():
    global audio_model
    audio_model = None; gc.collect()
    print(f"✓ AUDIO Whisper unloaded | TPU HBM {tpu_hbm_gb():.1f} GB")

def _transcribe_chunk(arr, lang, max_tokens=440):
    tok = audio_processor.tokenizer
    feats = audio_processor(arr, sampling_rate=SR, return_tensors="pt").input_features.to(torch.bfloat16)
    tokens = tok.convert_tokens_to_ids(["<|startoftranscript|>", f"<|{lang}|>", "<|transcribe|>", "<|notimestamps|>"])
    n_prompt, eot = len(tokens), tok.convert_tokens_to_ids("<|endoftext|>")
    with env, torch.no_grad():
        enc = audio_model.model.encoder(feats.to("jax")).last_hidden_state
        for _ in range(max_tokens):
            ids = torch.tensor([tokens], dtype=torch.long).to("jax")
            h = audio_model.model.decoder(input_ids=ids, encoder_hidden_states=enc,
                                          use_cache=False).last_hidden_state
            nxt = int(to_cpu(audio_model.proj_out(h))[0, -1].float().argmax())
            if nxt == eot:
                break
            tokens.append(nxt)
    return tok.decode(tokens[n_prompt:], skip_special_tokens=True).strip()

def transcribe(path, lang="en"):
    arr, _ = librosa.load(path, sr=SR)
    return " ".join(_transcribe_chunk(arr[i:i + CHUNK], lang) for i in range(0, max(len(arr), 1), CHUNK)).strip()

# ================= 3. VISION: Gemma-4 E4B via torchax (on demand) =================
VISION_REPO = "frankmorales2020/gemma-4-e4b-unesco-optimized"
vision_model = vision_processor = None

def load_vision():
    global vision_model, vision_processor
    vision_processor = AutoProcessor.from_pretrained(VISION_REPO)
    m = AutoModelForImageTextToText.from_pretrained(VISION_REPO, dtype=torch.bfloat16, device_map="cpu",
                                                    attn_implementation="eager")
    vision_model = m.dequantize().to(torch.bfloat16).eval()
    del m
    with env:
        vision_model.to("jax")
    gc.collect()
    print(f"✓ VISION Gemma-4 loaded | TPU HBM {tpu_hbm_gb():.1f} GB")

def unload_vision():
    global vision_model
    vision_model = None; gc.collect()
    print(f"✓ VISION Gemma-4 unloaded | TPU HBM {tpu_hbm_gb():.1f} GB")

def describe(image, prompt="Describe this image.", max_new_tokens=150):
    msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = vision_processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                                  return_dict=True, return_tensors="pt")
    n_in = inputs["input_ids"].shape[1]
    with env, torch.no_grad():
        tpu_in = {k: (v.to("jax") if torch.is_tensor(v) else v) for k, v in inputs.items()}
        out = to_cpu(vision_model.generate(**tpu_in, max_new_tokens=max_new_tokens, do_sample=False))
    return vision_processor.decode(out[0][n_in:], skip_special_tokens=True).strip()

# ================= helpers =================
def download(url, folder="/content/media"):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, url.split("/")[-1].split("?")[0] or "file")
    with requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path

first_sentence = lambda t: re.split(r"(?<=[.!?])\s", t.replace("\n", " ").strip())[0]

# ================= DEMO PIPELINE =================
AUDIO_URLS = [
    "https://huggingface.co/datasets/patrickvonplaten/audio_samples/resolve/main/bcn_weather.mp3",
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac",
]
IMAGE_URLS = [
    "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/bee_on_flower.jpg",
    "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/wisconsin_boardwalk.jpg",
]

print("\n" + "=" * 70 + "\n🎧 AUDIO → ENGLISH → HINDI\n" + "=" * 70)
load_audio()
texts = []
for url in AUDIO_URLS:
    t = time.time(); en = transcribe(download(url)); texts.append(en)
    print(f"🎧 {url.split('/')[-1]} ({time.time()-t:.0f}s)\n   EN: {en}")
unload_audio()
for en, hi in zip(texts, translate(texts)):
    print(f"   HI: {hi}")

print("\n" + "=" * 70 + "\n🖼️  IMAGE → ENGLISH → HINDI\n" + "=" * 70)
load_vision()
descs = []
for url in IMAGE_URLS:
    img = Image.open(download(url)).convert("RGB")
    t = time.time(); d = describe(img); descs.append(first_sentence(d))
    print(f"🖼️  {url.split('/')[-1]} ({time.time()-t:.0f}s)\n   EN: {descs[-1]}")
unload_vision()
for hi in translate(descs):
    print(f"   HI: {hi}")

print(f"\n✅ Done | TPU HBM now {tpu_hbm_gb():.1f} GB (Sarvam-1 stays loaded)")

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


✓ TEXT  Sarvam-1 on TPU V6E | TPU HBM 6.3 GB

🎧 AUDIO → ENGLISH → HINDI
✓ AUDIO Whisper loaded | TPU HBM 8.1 GB
🎧 bcn_weather.mp3 (89s)
   EN: Yesterday it was 35 degrees in Barcelona but today the temperature will go down to minus 20 degrees.
🎧 mlk.flac (21s)
   EN: I have a dream that one day this nation will rise up and live out the true meaning of its creed.
✓ AUDIO Whisper unloaded | TPU HBM 6.3 GB
   HI: कल बार्सिलोना में 35 डिग्री था लेकिन आज तापमान 20 डिग्री तक गिर जाएगा।
   HI: मैं चाहता हूं कि एक दिन यह राष्ट्र अपने सच्चे अर्थ में जाग उठे।

🖼️  IMAGE → ENGLISH → HINDI
✓ VISION Gemma-4 loaded | TPU HBM 22.4 GB
🖼️  bee_on_flower.jpg (757s)
   EN: This is a close-up photograph of a vibrant pink flower, likely a type of cosmos, in a garden setting.
🖼️  wisconsin_boardwalk.jpg (228s)
   EN: This is a vibrant, expansive landscape photograph dominated by a long, wooden boardwalk cutting through a lush, green field under a bright, dynamic sky.
✓ VISION Gemma-4 unloaded | TPU HBM 6.1 

## H2E SHERRIF

In [2]:
# ============================================================================
# CELL 3: H2E GOVERNANCE — WIRED TO THE 3 TPU MODELS
#   Text: Sarvam-1 (vLLM, TPU)   Audio: Whisper large-v3-turbo (torchax, TPU)
#   Vision: Gemma-4 E4B UNESCO fine-tune (torchax, TPU)
#   Requires the three-model pipeline cell to have been run first.
# ============================================================================
import hashlib, math, time
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from enum import Enum
import numpy as np, librosa, torch
from PIL import Image

# =============================================================================
# H2E CORE — LAMBDA SPECTRAL COMPLEMENTARITY THEOREM
# =============================================================================
PRIMES = [2, 3, 5, 7, 11, 13]

def compute_lambda_from_primes(primes: List[int] = PRIMES) -> float:
    """Lambda = 1 - prod_{p in primes} (1 - p^(-1/2))"""
    I = 1.0
    for p in primes:
        I *= (1.0 - 1.0 / math.sqrt(p))
    return 1.0 - I

LAMBDA = compute_lambda_from_primes()   # = 0.9785142874
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================================
# PART 0: LAMBDA (derived from primes)
# ============================================================================
class DynamicLambda:
    """Lambda = 1 - I, with I = prod_{p<=max_prime} (1 - p^(-1/2))."""
    def __init__(self, max_prime: int = 13):
        self.max_prime = max_prime

    def _get_primes_up_to(self, n: int) -> List[int]:
        if n < 2:
            return []
        sieve = [True] * (n + 1); sieve[0] = sieve[1] = False
        for p in range(2, int(n ** 0.5) + 1):
            if sieve[p]:
                for m in range(p * p, n + 1, p):
                    sieve[m] = False
        return [p for p, ok in enumerate(sieve) if ok]

    def compute_inverse_product(self) -> float:
        prod = 1.0
        for p in self._get_primes_up_to(self.max_prime):
            prod *= (1.0 - 1.0 / math.sqrt(p))
        return prod

    def compute(self) -> float:
        primes = self._get_primes_up_to(self.max_prime)
        I = self.compute_inverse_product()
        lam = compute_lambda_from_primes(primes)
        self.last_computation = {"primes": primes, "inverse_product": I, "lambda": lam,
                                 "formula": "Lambda = 1 - prod(1 - p^-1/2)"}
        return lam

    @property
    def value(self) -> float:
        return self.compute()

    def get_audit_hash(self) -> str:
        if not hasattr(self, "last_computation"):
            self.compute()
        c = self.last_computation
        return hashlib.sha256(f"lambda_{c['lambda']:.10f}_primes_{c['primes']}".encode()).hexdigest()[:16]

# ============================================================================
# PART 1: RIEMANNIAN GEOMETRY (unchanged)
# ============================================================================
class HyperbolicPlaneH2:
    @staticmethod
    def distance(z1: complex, z2: complex) -> float:
        z1, z2 = HyperbolicPlaneH2._to_disk(z1), HyperbolicPlaneH2._to_disk(z2)
        num = 2 * abs(z1 - z2) ** 2
        denom = max((1 - abs(z1) ** 2) * (1 - abs(z2) ** 2), 1e-8)
        return float(np.arccosh(max(1 + num / denom, 1.0)))

    @staticmethod
    def _to_disk(z: complex) -> complex:
        return z / (abs(z) + 1e-8) * 0.999 if abs(z) >= 1 else z

class SPD3Manifold:
    @staticmethod
    def distance(P: np.ndarray, Q: np.ndarray) -> float:
        P, Q = SPD3Manifold._make_spd(P), SPD3Manifold._make_spd(Q)
        try:
            ev, V = np.linalg.eigh(P)
            P_si = V @ np.diag(1.0 / np.sqrt(np.maximum(ev, 1e-6))) @ V.T
            evm, Vm = np.linalg.eigh(P_si @ Q @ P_si)
            logM = Vm @ np.diag(np.log(np.maximum(evm, 1e-8))) @ Vm.T
            return float(np.sqrt(np.trace(logM @ logM)))
        except Exception:
            return 2.0

    @staticmethod
    def _make_spd(M: np.ndarray) -> np.ndarray:
        ev, V = np.linalg.eigh((M + M.T) / 2)
        return V @ np.diag(np.maximum(ev, 0.1)) @ V.T

# ============================================================================
# PART 2-3: SPECTRAL MANIFOLD + L-EFM-AST OPERATOR (unchanged)
# ============================================================================
class EFMSpectralManifold:
    ZETA_ZEROS_IMAG_50 = [
        14.13, 21.02, 25.01, 30.42, 32.94, 37.59, 40.92, 43.33, 48.01, 49.77,
        52.97, 56.45, 59.35, 60.83, 65.11, 67.08, 69.55, 72.07, 75.70, 77.14,
        79.34, 82.91, 84.74, 87.43, 88.81, 92.49, 94.65, 95.87, 98.83, 101.32,
        103.73, 105.45, 107.17, 109.22, 111.03, 113.13, 114.95, 116.77, 118.57,
        120.00, 121.71, 123.08, 124.87, 126.81, 128.74, 129.92, 131.64, 133.21,
        134.85, 136.54]

    def __init__(self, dimension: int = 50, seed: int = SEED, lambda_value: float = None):
        self.dimension = min(dimension, len(self.ZETA_ZEROS_IMAG_50))
        self.LAMBDA = lambda_value if lambda_value is not None else DynamicLambda().compute()
        z = self.ZETA_ZEROS_IMAG_50[:self.dimension]
        self.normalized_zeros = np.array([0.5 + 0.5 * (g - z[0]) / (z[-1] - z[0]) for g in z])
        rng = np.random.RandomState(seed)
        Q, _ = np.linalg.qr(rng.randn(self.dimension, self.dimension))
        H = Q @ np.diag(self.normalized_zeros) @ Q.T
        self.H = (H + H.T) / 2

    def project(self, e: np.ndarray) -> np.ndarray:
        return self.H @ e if e.ndim == 1 else e @ self.H.T

    def spectral_alignment(self, z: np.ndarray, w: np.ndarray) -> float:
        Hz = self.project(z)
        nz, nw = np.linalg.norm(Hz), np.linalg.norm(w)
        if nz < 1e-8 or nw < 1e-8:
            return 0.0
        cosine = (np.dot(Hz, w) / (nz * nw) + 1.0) / 2.0
        return float(max(0.0, min(1.0, cosine * self.LAMBDA)))

class LEFMASTOperator:
    def __init__(self, efm: EFMSpectralManifold, lambda_value: float = None):
        self.efm = efm
        self.LAMBDA = lambda_value if lambda_value is not None else DynamicLambda().compute()

    def compute_spectral_sroi(self, z: np.ndarray, w: np.ndarray) -> float:
        return self.efm.spectral_alignment(z, w)

# ============================================================================
# PART 4: DECISION ENGINE (content-based features)
# ============================================================================
class GenerationMode(Enum):
    SAFE = "safe"
    REJECTED = "rejected"
    SPECTRAL_GUARANTEED = "spectral_guaranteed"

@dataclass
class H2EResponse:
    accepted: bool
    final_sroi: float
    geometric_sroi: float
    spectral_sroi: float
    lefm_sroi: float
    generation_mode: GenerationMode
    response_text: Optional[str]
    geodesic_distance: float
    energy_estimate: float
    deterministic_hash: str
    modalities_used: List[str]
    lefm_pass: bool
    lambda_used: float
    lambda_audit_hash: str
    transcript: Optional[str] = None
    description: Optional[str] = None
    hindi: Dict[str, str] = field(default_factory=dict)
    timings_s: Dict[str, float] = field(default_factory=dict)

class H2EDecisionEngine:
    DIM = 50

    def __init__(self, strategy: str = "geometric_only", max_prime: int = 13):
        self.strategy = strategy
        self.lambda_calculator = DynamicLambda(max_prime=max_prime)
        self.LAMBDA = self.THRESHOLD = self.lambda_calculator.compute()
        self.safe_h2_ref = complex(0.0, 0.0)
        self.safe_spd3_ref = np.eye(3)
        self.SCALE = 50.0
        self.efm = EFMSpectralManifold(dimension=self.DIM, seed=SEED, lambda_value=self.LAMBDA)
        self.lefm_ast = LEFMASTOperator(self.efm, lambda_value=self.LAMBDA)
        self.text_energy_per_token, self.audio_energy_per_sec, self.vision_energy_per_inference = 0.6132, 0.5, 124.0
        self.metrics_history = []
        c = self.lambda_calculator.last_computation
        print(f"\n{'='*70}\nH2E DECISION ENGINE INITIALIZED\n{'='*70}")
        print(f"  Strategy:           {strategy}")
        print(f"  Primes:             {c['primes']}")
        print(f"  I = prod(1-p^-1/2): {c['inverse_product']:.10f}")
        print(f"  Lambda = 1 - I:     {self.LAMBDA:.10f}")
        print("=" * 70)

    @staticmethod
    def _normalize(v: np.ndarray) -> np.ndarray:
        v = v - v.mean()
        n = np.linalg.norm(v)
        return v / n if n > 1e-8 else np.ones_like(v) / np.sqrt(len(v))

    def _text_features(self, text: str) -> np.ndarray:
        v = np.zeros(self.DIM)
        t = text.lower()
        for g in t.split() + [t[i:i + 3] for i in range(max(len(t) - 2, 0))]:
            h = int(hashlib.md5(g.encode()).hexdigest(), 16)
            v[h % self.DIM] += 1.0 if (h >> 8) & 1 else -1.0
        return self._normalize(v)

    def _vision_features(self, image: Image.Image) -> np.ndarray:
        a = np.asarray(image.convert("L").resize((10, 5)), dtype=np.float64).reshape(-1) / 255.0
        return self._normalize(a)

    def _audio_features(self, audio: np.ndarray) -> np.ndarray:
        spec = np.abs(np.fft.rfft(audio.astype(np.float64))) if len(audio) else np.zeros(self.DIM)
        bands = np.array_split(spec, self.DIM)
        return self._normalize(np.log1p(np.array([b.mean() if len(b) else 0.0 for b in bands])))

    def _embedding_to_h2(self, e: np.ndarray) -> complex:
        theta = np.sum(e[:2]) % (2 * np.pi)
        r = 0.5 * np.tanh(np.linalg.norm(e[:5]))
        return complex(r * np.cos(theta), r * np.sin(theta))

    def _embeddings_to_spd3(self, t, a, v) -> np.ndarray:
        g = lambda e, i: 0.1 if e is None or len(e) == 0 else float(e[i % len(e)])
        M = np.array([[1.0 + g(t, 0), g(a, 0), g(v, 0)],
                      [g(a, 0), 1.0 + g(a, 1), g(v, 1)],
                      [g(v, 0), g(v, 1), 1.0 + g(v, 2)]])
        return SPD3Manifold._make_spd(M)

    def compute_geometric_sroi(self, t, a, v) -> float:
        pts = [self._embedding_to_h2(e) for e in (t, a, v) if e is not None]
        if not pts:
            return 0.0
        h2 = np.mean([HyperbolicPlaneH2.distance(p, self.safe_h2_ref) for p in pts])
        z3 = np.zeros(3)
        spd = SPD3Manifold.distance(self._embeddings_to_spd3(
            t if t is not None else z3, a if a is not None else z3, v if v is not None else z3),
            self.safe_spd3_ref)
        return float(np.exp(-np.sqrt(h2 ** 2 + spd ** 2) / self.SCALE))

    def decide(self, text=None, audio=None, vision=None, strategy=None) -> H2EResponse:
        strategy = strategy or self.strategy
        energy, used = 0.0, []
        t = a = v = None
        if text:
            t = self._text_features(text); used.append("text")
            energy += len(text.split()) * self.text_energy_per_token
        if audio is not None:
            a = self._audio_features(audio); used.append("audio")
            energy += len(audio) / 16000 * self.audio_energy_per_sec
        if vision is not None:
            v = self._vision_features(vision); used.append("vision")
            energy += self.vision_energy_per_inference

        geo = self.compute_geometric_sroi(t, a, v)
        parts = [e for e in (t, a, v) if e is not None]
        intent = np.mean(parts, axis=0) if parts else np.ones(self.DIM) / np.sqrt(self.DIM)
        state = v if v is not None else (t if t is not None else np.ones(self.DIM) / np.sqrt(self.DIM))
        spec = self.efm.spectral_alignment(intent, state)
        lefm = self.lefm_ast.compute_spectral_sroi(intent, state)
        self.metrics_history.append({"geometric": geo, "spectral": spec, "lefm_ast": lefm})

        gp, sp, lp = geo >= self.THRESHOLD, spec >= self.THRESHOLD, lefm >= self.THRESHOLD
        if strategy == "conservative":
            accepted = gp and sp and lp
            mode = GenerationMode.SPECTRAL_GUARANTEED if accepted else GenerationMode.REJECTED
        else:
            accepted = gp
            mode = GenerationMode.SAFE if accepted else GenerationMode.REJECTED

        dist = -self.SCALE * math.log(geo) if geo > 1e-8 else 10.0
        h = hashlib.sha256(f"{text}{accepted}{geo:.10f}{spec:.10f}{lefm:.10f}{used}{self.LAMBDA:.10f}"
                           .encode()).hexdigest()[:16]
        verdict = "ACCEPTED" if accepted else "REJECTED"
        return H2EResponse(
            accepted=accepted, final_sroi=geo if accepted else 0.0,
            geometric_sroi=geo, spectral_sroi=spec, lefm_sroi=lefm, generation_mode=mode,
            response_text=f"[H2E] G={geo:.4f} S={spec:.4f} L={lefm:.4f} | Lambda={self.LAMBDA:.4f} | {verdict}",
            geodesic_distance=dist, energy_estimate=energy, deterministic_hash=h,
            modalities_used=used, lefm_pass=lp, lambda_used=self.LAMBDA,
            lambda_audit_hash=self.lambda_calculator.get_audit_hash())

    def get_statistics(self) -> Dict:
        if not self.metrics_history:
            return {"error": "No decisions made yet"}
        acc = sum(m["geometric"] >= self.THRESHOLD for m in self.metrics_history)
        return {"total_decisions": len(self.metrics_history), "accepted_count": acc,
                "acceptance_rate": acc / len(self.metrics_history),
                "avg_geometric": float(np.mean([m["geometric"] for m in self.metrics_history])),
                "avg_spectral": float(np.mean([m["spectral"] for m in self.metrics_history])),
                "avg_lefm_ast": float(np.mean([m["lefm_ast"] for m in self.metrics_history])),
                "lambda": self.LAMBDA, "lambda_audit_hash": self.lambda_calculator.get_audit_hash()}

# ============================================================================
# PART 5: GOVERNANCE WIRED TO THE 3 MODELS
# ============================================================================
class H2ECompleteGovernance:
    def __init__(self, strategy="geometric_only", max_prime=13, translate_outputs=True):
        self.decision_engine = H2EDecisionEngine(strategy=strategy, max_prime=max_prime)
        self.translate_outputs = translate_outputs
        print(f"\n{'='*70}\nH2E GOVERNANCE — 3 TPU MODELS\n{'='*70}")
        print("  Text:   Sarvam-1 (vLLM, resident)")
        print("  Audio:  Whisper large-v3-turbo (torchax, on demand)")
        print("  Vision: Gemma-4 E4B UNESCO fine-tune (torchax, on demand)")
        print("=" * 70)

    def _run_audio(self, audio: np.ndarray) -> str:
        load_audio()
        try:
            return " ".join(_transcribe_chunk(audio[i:i + CHUNK], "en")
                            for i in range(0, max(len(audio), 1), CHUNK)).strip()
        finally:
            unload_audio()

    def _run_vision(self, image: Image.Image) -> str:
        load_vision()
        try:
            return describe(image)
        finally:
            unload_vision()

    def process(self, text: str = None, audio: np.ndarray = None, vision: Image.Image = None) -> H2EResponse:
        timings, transcript, description = {}, None, None
        if audio is not None:
            t0 = time.time(); transcript = self._run_audio(audio); timings["whisper"] = time.time() - t0
        if vision is not None:
            t0 = time.time(); description = self._run_vision(vision); timings["gemma4"] = time.time() - t0

        combined = " ".join(s for s in (text, transcript, description) if s) or None
        resp = self.decision_engine.decide(text=combined, audio=audio, vision=vision)
        resp.transcript, resp.description = transcript, description

        if resp.accepted and self.translate_outputs:
            items = {k: s for k, s in (("text", text), ("transcript", transcript),
                                       ("description", first_sentence(description) if description else None)) if s}
            t0 = time.time()
            resp.hindi = dict(zip(items, translate(list(items.values()))))
            timings["sarvam1"] = time.time() - t0
        resp.timings_s = timings
        return resp

# ============================================================================
# DEMO — real inputs
# ============================================================================
print("\n" + "=" * 80 + "\nH2E GOVERNANCE — DEMONSTRATION WITH REAL INPUTS\n" + "=" * 80)
h2e = H2ECompleteGovernance(strategy="geometric_only", max_prime=13)

audio_path = download("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac")
audio_arr, _ = librosa.load(audio_path, sr=SR)
image = Image.open(download(
    "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/wisconsin_boardwalk.jpg")).convert("RGB")

test_cases = [
    {"name": "Text only",            "text": "What is the capital of France?", "audio": None,      "vision": None},
    {"name": "Audio (speech)",       "text": None,                             "audio": audio_arr, "vision": None},
    {"name": "Vision (photo)",       "text": "Describe this landscape",        "audio": None,      "vision": image},
    {"name": "All three modalities", "text": "Analyze this scene",             "audio": audio_arr, "vision": image},
]

for case in test_cases:
    print(f"\n{'='*60}\nTest: {case['name']}\n{'='*60}")
    r = h2e.process(text=case["text"], audio=case["audio"], vision=case["vision"])
    print(f"  Decision:        {'ACCEPTED' if r.accepted else 'REJECTED'}  ({', '.join(r.modalities_used)})")
    print(f"  Geometric SROI:  {r.geometric_sroi:.6f}")
    print(f"  Spectral SROI:   {r.spectral_sroi:.6f}")
    print(f"  L-EFM-AST SROI:  {r.lefm_sroi:.6f}")
    print(f"  Threshold:       {r.lambda_used:.10f}")
    if r.transcript:  print(f"  Whisper:         {r.transcript}")
    if r.description: print(f"  Gemma-4:         {first_sentence(r.description)}")
    for k, hi in r.hindi.items():
        print(f"  Hindi ({k}): {hi}")
    print(f"  Energy (est.):   {r.energy_estimate:.2f}")
    if r.timings_s:
        print(f"  Timings:         " + ", ".join(f"{k} {v:.0f}s" for k, v in r.timings_s.items()))
    print(f"  Hash:            {r.deterministic_hash}")

print("\n" + "=" * 80 + "\nSTATISTICS\n" + "=" * 80)
for k, v in h2e.decision_engine.get_statistics().items():
    print(f"  {k:<18} {v}")

# ============================================================================
# SUMMARY (what is actually running)
# ============================================================================
print(f"""
{'='*80}
H2E ARCHITECTURE — AS DEPLOYED IN THIS NOTEBOOK (TPU v6e, 31 GB HBM)
{'='*80}
LAYER 0: Lambda = 1 - prod_(p in {PRIMES}) (1 - p^-1/2) = {LAMBDA:.10f}

LAYER 1-2: MODELS (all on the TPU)
  - Text:   Sarvam-1 2B, vLLM 0.21, resident (~6 GB HBM)
  - Audio:  Whisper large-v3-turbo, torchax, loaded on demand (~2 GB HBM)
  - Vision: Gemma-4 E4B UNESCO fine-tune, torchax, on demand (~16 GB HBM)
            benchmark quality 0.983 on the 3-image UNESCO set

LAYER 3: THREE METRICS on content-derived features
  - Geometric SROI (H^2 x SPD(3))
  - Spectral SROI (zeta-zero spectral manifold)
  - L-EFM-AST SROI

LAYER 4: DECISION ENGINE
  - geometric_only: geometric metric >= Lambda
  - conservative:   all three metrics >= Lambda

DETERMINISTIC: greedy decoding, seed {SEED}; SHA-256 hash per decision.
NOTE: the SROI scores measure geometric similarity to a reference point;
      they are not a validated safety classifier.
""")


H2E GOVERNANCE — DEMONSTRATION WITH REAL INPUTS

H2E DECISION ENGINE INITIALIZED
  Strategy:           geometric_only
  Primes:             [2, 3, 5, 7, 11, 13]
  I = prod(1-p^-1/2): 0.0214857126
  Lambda = 1 - I:     0.9785142874

H2E GOVERNANCE — 3 TPU MODELS
  Text:   Sarvam-1 (vLLM, resident)
  Audio:  Whisper large-v3-turbo (torchax, on demand)
  Vision: Gemma-4 E4B UNESCO fine-tune (torchax, on demand)

Test: Text only
  Decision:        ACCEPTED  (text)
  Geometric SROI:  0.997282
  Spectral SROI:   0.974133
  L-EFM-AST SROI:  0.974133
  Threshold:       0.9785142874
  Hindi (text): फ्रांस की राजधानी पेरिस है।
  Energy (est.):   3.68
  Timings:         sarvam1 1s
  Hash:            6433891743cff188

Test: Audio (speech)
✓ AUDIO Whisper loaded | TPU HBM 8.1 GB
✓ AUDIO Whisper unloaded | TPU HBM 6.1 GB
  Decision:        ACCEPTED  (text, audio)
  Geometric SROI:  0.992468
  Spectral SROI:   0.834043
  L-EFM-AST SROI:  0.834043
  Threshold:       0.9785142874
  Whisper:         I 